# Phase 6 - Load cleaned data into PostgreSQL

Creates the `te_analysis` database (if missing), runs the schema, and loads all cleaned
tables plus the trip-level `trip_analytics` table.

**Security note:** do not hard-code your password if you push to GitHub. This notebook reads
it from an environment variable `PG_PASSWORD`. For a quick local run you can paste it into
`PASSWORD` below, but delete it before committing.

## Step 0 - Install the driver (run once)
SQLAlchemy + psycopg2 let pandas talk to PostgreSQL.

In [ ]:
# Run once if needed, then you can comment it out.
# %pip install sqlalchemy psycopg2-binary

In [ ]:
import os
import pandas as pd
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text

HOST = 'localhost'
PORT = '5432'
USER = 'postgres'
DB   = 'te_analysis'

# Read the password from an environment variable so no secret is stored in this file.
# Before running, set it once in your terminal:  setx PG_PASSWORD "your_password"  (then reopen Jupyter)
PASSWORD = os.environ.get('PG_PASSWORD')
if not PASSWORD:
    raise ValueError('Set the PG_PASSWORD environment variable before running (see note above).')

# IMPORTANT: url-encode user & password so special characters like @ : / do not break the URL.
USER_Q = quote_plus(USER)
PW_Q = quote_plus(PASSWORD)

PROC = r'c:\Users\Sweta\OneDrive\Desktop\proj2\data\processed'
SQL_DIR = r'c:\Users\Sweta\OneDrive\Desktop\proj2\sql'
print('config set')

## Step 1 - Create the database te_analysis (if it does not exist)
We connect to the default 'postgres' database first, then create ours.

In [ ]:
admin = create_engine(f'postgresql+psycopg2://{USER_Q}:{PW_Q}@{HOST}:{PORT}/postgres')
with admin.connect() as conn:
    conn.execution_options(isolation_level='AUTOCOMMIT')
    exists = conn.execute(text("SELECT 1 FROM pg_database WHERE datname = :d"), {'d': DB}).fetchone()
    if not exists:
        conn.execute(text(f'CREATE DATABASE {DB}'))
        print('created database', DB)
    else:
        print('database already exists:', DB)

## Step 2 - Run the schema (create tables)

In [ ]:
engine = create_engine(f'postgresql+psycopg2://{USER_Q}:{PW_Q}@{HOST}:{PORT}/{DB}')
with open(os.path.join(SQL_DIR, '01_schema.sql'), 'r') as f:
    schema_sql = f.read()
with engine.begin() as conn:
    conn.execute(text(schema_sql))
print('schema created')

## Step 3 - Load the cleaned CSVs into the tables
We empty the tables first (safe to re-run), then load in FK order (parents first): offices, vendors, employees, trips, expenses.

In [ ]:
# Empty all tables first so this cell is safe to re-run. Child tables before parents,
# and CASCADE handles the FK order in one shot.
with engine.begin() as conn:
    conn.execute(text('TRUNCATE expenses, trips, employees, vendors, offices RESTART IDENTITY CASCADE'))
print('tables emptied')

def load(csv_name, table):
    df = pd.read_csv(os.path.join(PROC, csv_name))
    df.to_sql(table, engine, if_exists='append', index=False)
    print(f'loaded {table:16s} rows={len(df)}')

load('offices_clean.csv', 'offices')
load('vendors_clean.csv', 'vendors')
load('employees_clean.csv', 'employees')

# trips_clean has extra helper columns; keep only the schema columns
trips = pd.read_csv(os.path.join(PROC, 'trips_clean.csv'))
trip_cols = ['trip_id','employee_id','client_project_code','destination_city','trip_purpose',
             'booking_channel','vendor_id','booked_date','trip_start_date','trip_end_date','trip_status']
trips = trips[trip_cols].copy()
# vendor_id is float64 (has 115 blanks). Convert to a nullable integer so Postgres
# gets 16 and NULL, NOT 16.0 and NaN, which an INT column rejects.
trips['vendor_id'] = trips['vendor_id'].astype('Int64')
trips.to_sql('trips', engine, if_exists='append', index=False)
print('loaded trips           rows=', len(trips))

exp = pd.read_csv(os.path.join(PROC, 'expenses_clean.csv'))
exp_cols = ['line_item_id','trip_id','category','amount','currency','submitted_date',
            'approval_status','is_policy_exception','exception_reason','approved_by_level','is_approved']
exp = exp[exp_cols].copy()
# same safety: keys as nullable integers
exp['line_item_id'] = exp['line_item_id'].astype('Int64')
exp['trip_id'] = exp['trip_id'].astype('Int64')
exp.to_sql('expenses', engine, if_exists='append', index=False)
print('loaded expenses        rows=', len(exp))

## Step 4 - Load the trip-level analytical table

In [ ]:
ta = pd.read_csv(os.path.join(PROC, 'trip_analytics.csv'))
ta.to_sql('trip_analytics', engine, if_exists='replace', index=False)
print('loaded trip_analytics rows=', len(ta))

## Step 5 - Verify row counts in the database

In [ ]:
with engine.connect() as conn:
    for t in ['offices','vendors','employees','trips','expenses','trip_analytics']:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
        print(f'{t:16s} {n}')